# WaZP Lensing Signals

Produce per-cluster ΔΣ data vectors for the WaZP "real-data" catalog built in `wazp_cluster_selection.ipynb`. Pipeline:

1. **Load WaZP real-data catalog** (§1) from `data/wazp_real_data.fits`.
2. **One-time pre-filter** of the 291 GB DES Y3 metacal HDF5 + 6 GB DNF photo-z HDF5 (§2). Apply the standard metacal cuts (`flags==0 & snr ∈ [10,1000] & size_ratio > 0.5 & T > 0`), require `zmean_sof > 0`, keep only the 10 columns `xlensing.data.metacal_cluster_lensing` needs, save to `data/metacal_filtered.npz`. Subsequent runs reload the intermediate.
3. **BallTree** over source (DEC, RA) in radians, haversine metric, for fast per-cluster cone queries (§3).
4. **Radial-bin layout** following the `prototype_destacking` convention (§4).
5. **Worker function** (§5): per cluster, query the cone, call `metacal_cluster_lensing`, package its output, call `single_cluster`, return `(sigmas, sigmas_cov, xigmas, xigmas_cov)`.
6. **Smoke-test on one cluster** (§6).
7. **`multiprocess.Pool`** over all WaZP clusters (§7).
8. **Save & sample plots** (§8).

Photo-z handling: the DNF file `DESY3_GOLD_2_2.1_DNF.h5` is row-aligned with metacal (same N = 399 263 026, same `coadd_object_id` ordering). We use `zmean_sof` as the per-source point estimate.

In [ ]:
import os, time, pickle, warnings
import numpy as np
import h5py
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table
from astropy.cosmology import Planck18 as COSMO
from astropy import units as u
from sklearn.neighbors import BallTree
from multiprocess import Pool, cpu_count
import xlensing.data
warnings.filterwarnings('ignore')

DATA          = '/mnt/c/Users/andre/DATA'
METACAL_FILE  = f'{DATA}/DESY3_metacal_v03-004.h5'
DNF_FILE      = f'{DATA}/DESY3_GOLD_2_2.1_DNF.h5'
WAZP_REAL     = 'data/wazp_real_data.fits'
FILTERED_NPZ  = 'data/metacal_filtered.npz'
RESULTS_PKL   = 'data/wazp_lensing_signals.pkl'

# Pipeline parameters --------------------------------------------------------
USE_CACHE  = True          # set to False to force a clean rebuild of every intermediate
RADIUS_MPC = 5.0           # Mpc, cone radius passed to metacal_cluster_lensing
NBOOT      = 200           # bootstrap resamplings for single_cluster cov
NWORKERS   = max(1, cpu_count() // 2)
CHUNK_READ = 10_000_000    # metacal rows per chunk during pre-filter

# Radial bins (Mpc), mirroring notebooks/prototype_destacking convention -----
radii  = np.logspace(-0.8, 0.6, 8)
_N     = len(radii)
_edges = np.logspace(
    np.log10(radii[0])   + (np.log10(radii[0]) - np.log10(radii[1])) / 2,
    np.log10(radii[-1])  - (np.log10(radii[0]) - np.log10(radii[1])) / 2,
    _N + 1,
)
BINS_LIMS = np.array([[_edges[i], _edges[i+1]] for i in range(_N)])

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
os.makedirs('data', exist_ok=True)

print(f'USE_CACHE  = {USE_CACHE}')
print(f'NWORKERS   = {NWORKERS}')
print(f'CHUNK_READ = {CHUNK_READ:,}')
print(f'RADIUS_MPC = {RADIUS_MPC}')
print(f'NBOOT      = {NBOOT}')
print(f'Radial bin centres (Mpc): {np.round(radii, 3)}')

## 1 — Load the WaZP real-data catalog

In [ ]:
wazp_all = Table.read(WAZP_REAL)
print(f'Loaded {len(wazp_all):,} clusters from {WAZP_REAL}')

# Same selection as wazp_mass_richness_by_delta_m: drop clusters with
# non-finite or non-positive DELTA_M_I — these are cluster-finder
# artefacts where the catalog's BCG is fainter than the brightest
# inner satellite.  Upstream wazp_cluster_selection already imposed
# NGALS > 25, ZPHOT < 0.7, and BCG-present.
keep = np.isfinite(wazp_all['DELTA_M_I']) & (wazp_all['DELTA_M_I'] > 0)
wazp = wazp_all[keep]
print(f'After Δm > 0 cut: {len(wazp):,}  '
      f'(dropped {len(wazp_all) - len(wazp):,})')
print(f'Columns: {wazp.colnames}')
print(wazp[:5])


## 2 — One-time pre-filter of the metacal + DNF catalogs

Streams both HDF5 files in `CHUNK_READ`-row chunks, applies the standard metacal selection plus `zmean_sof > 0`, and keeps only the 10 columns that `metacal_cluster_lensing` consumes. Saves to `data/metacal_filtered.npz` so re-runs skip this step.

Row alignment is verified up-front via the first/last `coadd_object_id` in each file.

In [ ]:
if os.path.exists(FILTERED_NPZ) and USE_CACHE:
    print(f'Loading cached pre-filter from {FILTERED_NPZ}...')
    f = np.load(FILTERED_NPZ)
    SR_RA  = f['ra']
    SR_DEC = f['dec']
    SR_Z   = f['z']
    SR_E1  = f['e1']
    SR_E2  = f['e2']
    SR_W   = f['w']
    SR_R11 = f['R11']
    SR_R12 = f['R12']
    SR_R21 = f['R21']
    SR_R22 = f['R22']
    print(f'Loaded {len(SR_RA):,} pre-filtered sources')
else:
    if os.path.exists(FILTERED_NPZ):
        print(f'USE_CACHE = False, rebuilding {FILTERED_NPZ} from scratch')
    print(f'Building pre-filter from {METACAL_FILE} + {DNF_FILE} ...')
    cols = {k: [] for k in ['ra','dec','z','e1','e2','w',
                            'R11','R12','R21','R22']}
    with h5py.File(METACAL_FILE, 'r') as fm, h5py.File(DNF_FILE, 'r') as fz:
        g  = fm['catalog/unsheared']
        gz = fz['catalog/unsheared']
        Ntot = g['ra'].shape[0]
        assert Ntot == gz['zmean_sof'].shape[0], 'Row count mismatch'
        assert int(g['coadd_object_id'][0])      == int(gz['coadd_object_id'][0])
        assert int(g['coadd_object_id'][Ntot-1]) == int(gz['coadd_object_id'][Ntot-1])

        t0 = time.time()
        for s in range(0, Ntot, CHUNK_READ):
            e = min(s + CHUNK_READ, Ntot)

            flags = g['flags'][s:e]
            snr   = g['snr'][s:e]
            sr    = g['size_ratio'][s:e]
            T     = g['T'][s:e]
            zmean = gz['zmean_sof'][s:e]

            sel = ((flags == 0) & (snr > 10) & (snr < 1000)
                   & (sr > 0.5) & (T > 0) & (zmean > 0))
            if not sel.any():
                continue
            idx = np.where(sel)[0]

            cols['ra' ].append(np.deg2rad(g['ra' ][s:e][idx]).astype('float64'))
            cols['dec'].append(np.deg2rad(g['dec'][s:e][idx]).astype('float64'))
            cols['z'  ].append(zmean[idx].astype('float32'))
            cols['e1' ].append(g['e_1'][s:e][idx].astype('float32'))
            cols['e2' ].append(g['e_2'][s:e][idx].astype('float32'))
            cols['w'  ].append(g['weight'][s:e][idx].astype('float32'))
            cols['R11'].append(g['R11'][s:e][idx].astype('float32'))
            cols['R12'].append(g['R12'][s:e][idx].astype('float32'))
            cols['R21'].append(g['R21'][s:e][idx].astype('float32'))
            cols['R22'].append(g['R22'][s:e][idx].astype('float32'))

            dt   = time.time() - t0
            kept = sum(len(x) for x in cols['ra'])
            eta  = dt * (Ntot - e) / max(e, 1)
            print(f'  {e:>11,}/{Ntot:,}  kept {kept:>11,}  '
                  f'elapsed {dt:6.1f}s  eta {eta:6.1f}s', flush=True)

    print('Concatenating per-chunk buffers ...')
    cols = {k: np.concatenate(v) for k, v in cols.items()}
    print(f'Total kept: {len(cols["ra"]):,}')

    print(f'Writing {FILTERED_NPZ} ...')
    np.savez(FILTERED_NPZ,
        ra=cols['ra'], dec=cols['dec'], z=cols['z'],
        e1=cols['e1'], e2=cols['e2'], w =cols['w'],
        R11=cols['R11'], R12=cols['R12'],
        R21=cols['R21'], R22=cols['R22'],
    )
    SR_RA  = cols['ra'];  SR_DEC = cols['dec']; SR_Z = cols['z']
    SR_E1  = cols['e1'];  SR_E2  = cols['e2'];  SR_W = cols['w']
    SR_R11 = cols['R11']; SR_R12 = cols['R12']
    SR_R21 = cols['R21']; SR_R22 = cols['R22']
    print('Done.')

print()
print(f'Source count: {len(SR_RA):,}')
print(f'RA range  (rad): {SR_RA.min():.3f} to {SR_RA.max():.3f}')
print(f'DEC range (rad): {SR_DEC.min():.3f} to {SR_DEC.max():.3f}')
print(f'z range        : {SR_Z.min():.3f} to {SR_Z.max():.3f}')

## 3 — BallTree spatial index

`sklearn.neighbors.BallTree` with the haversine metric expects positions as `(lat, lon) = (dec, ra)` in radians. The tree is built once in the parent process; child workers inherit it through `fork()`.

In [ ]:
t0 = time.time()
positions = np.column_stack([SR_DEC, SR_RA])
TREE = BallTree(positions, metric='haversine')
print(f'BallTree built over {len(positions):,} sources in {time.time()-t0:.1f}s')

## 4 — Radial bins

In [ ]:
print('Radial bin layout (Mpc, proper):')
for i, (lo, hi) in enumerate(BINS_LIMS):
    print(f'  bin {i}: [{lo:.3f}, {hi:.3f}]  centre = {radii[i]:.3f}')

## 5 — Per-cluster worker

For each cluster:
1. Compute the worst-case angular cone radius `RADIUS_MPC / D_A(z_cl)`.
2. Query the BallTree for source indices inside that cone.
3. Build the 10-tuple in the order `metacal_cluster_lensing` expects (`(ra, dec, z, e1, e2, w, R11, R12, R21, R22)`, RA/DEC in radians).
4. Run `metacal_cluster_lensing` → repackage to the 6-row array shape that `single_cluster` expects → run `single_cluster([bkg], BINS_LIMS, NBOOT)`.

Source arrays and the BallTree live as module-level globals so forked workers inherit them via copy-on-write — no per-worker pickling cost.

In [ ]:
def cluster_worker(row):
    """row: dict with keys 'ID', 'RA' (deg), 'DEC' (deg), 'ZPHOT'."""
    try:
        cid   = int(row['ID'])
        z_cl  = float(row['ZPHOT'])
        ra_r  = np.deg2rad(float(row['RA']))
        dec_r = np.deg2rad(float(row['DEC']))

        D_A = float(COSMO.angular_diameter_distance(z_cl).to_value(u.Mpc))
        if D_A <= 0:
            return {'ID': cid, 'error': f'D_A={D_A}'}
        ang_r = RADIUS_MPC / D_A   # radians

        idx = TREE.query_radius(np.array([[dec_r, ra_r]]), r=ang_r)[0]
        if len(idx) == 0:
            return {'ID': cid, 'sigmas': None, 'count': 0}

        sources = (
            SR_RA [idx], SR_DEC[idx], SR_Z [idx],
            SR_E1 [idx], SR_E2 [idx], SR_W [idx],
            SR_R11[idx], SR_R12[idx], SR_R21[idx], SR_R22[idx],
        )
        cluster = (ra_r, dec_r, z_cl)

        res = xlensing.data.metacal_cluster_lensing(
            cluster, sources, radius=RADIUS_MPC,
        )
        if res['Count'] == 0:
            return {'ID': cid, 'sigmas': None, 'count': 0}

        bkg = np.array([
            np.asarray(res['Critical Density']),
            np.asarray(res['Tangential Shear']),
            np.asarray(res['Cross Shear']),
            np.asarray(res['Weights']),
            np.asarray(res['Radial Distance']),
            np.asarray(res['Mult. Bias']),
        ])

        # single_cluster now returns a 6-tuple with boost-corrected ΔΣ
        # and the full per-bin boost factor + covariances.  Mirrors the
        # stacked_signal API; bootstrap is over galaxies inside this
        # cluster's cone.
        sigmas, boosts, boosts_cov, sigmas_cov, xigmas, xigmas_cov = (
            xlensing.data.single_cluster([bkg], BINS_LIMS, NBOOT)
        )
        return {
            'ID':         cid,
            'sigmas':     np.asarray(sigmas),
            'sigmas_cov': np.asarray(sigmas_cov),
            'boosts':     np.asarray(boosts),
            'boosts_cov': np.asarray(boosts_cov),
            'xigmas':     np.asarray(xigmas),
            'xigmas_cov': np.asarray(xigmas_cov),
            'count':      int(res['Count']),
        }
    except Exception as exc:
        return {'ID': int(row.get('ID', -1)), 'error': repr(exc)}


## 6 — Smoke-test on a single cluster

Run one cluster serially before launching the pool to verify the pipeline end-to-end.

In [ ]:
row0 = {'ID': int(wazp['ID'][0]),  'RA': float(wazp['RA'][0]),
        'DEC': float(wazp['DEC'][0]), 'ZPHOT': float(wazp['ZPHOT'][0])}
t0 = time.time()
r0 = cluster_worker(row0)
dt = time.time() - t0
print(f'One cluster in {dt:.2f}s -> keys = {list(r0.keys())}')
if 'sigmas' in r0 and r0['sigmas'] is not None:
    print(f'  ΔΣ            = {r0["sigmas"]}')
    print(f'  diag σ_ΔΣ     = {np.sqrt(np.diag(r0["sigmas_cov"]))}')
    print(f'  boost B(R)    = {r0["boosts"]}')
    print(f'  diag σ_B(R)   = {np.sqrt(np.diag(r0["boosts_cov"]))}')
    print(f'  N srcs in cone = {r0["count"]}')


## 7 — Parallel run over all WaZP clusters

Linux fork lets each worker share the `SR_*` arrays and `TREE` read-only via COW, so there's no per-task pickling cost for the catalog. We use `imap_unordered` with a small `chunksize` so progress is visible and a slow cluster doesn't stall a batch.

In [ ]:
input_list = [
    {'ID': int(wazp['ID'][i]),    'RA':   float(wazp['RA'][i]),
     'DEC':float(wazp['DEC'][i]), 'ZPHOT':float(wazp['ZPHOT'][i])}
    for i in range(len(wazp))
]
print(f'Submitting {len(input_list):,} clusters to a pool of {NWORKERS} workers')

results = []
CKPT_EVERY = 1000
t0 = time.time()
with Pool(NWORKERS) as pool:
    for i, r in enumerate(pool.imap_unordered(
            cluster_worker, input_list, chunksize=4)):
        results.append(r)
        if (i + 1) % 200 == 0 or (i + 1) == len(input_list):
            dt  = time.time() - t0
            eta = dt * (len(input_list) - i - 1) / (i + 1)
            print(f'  {i+1:>6,}/{len(input_list):,}  '
                  f'elapsed {dt/60:5.1f} min  eta {eta/60:5.1f} min',
                  flush=True)
        if (i + 1) % CKPT_EVERY == 0:
            with open(RESULTS_PKL + '.partial', 'wb') as fh:
                pickle.dump(results, fh)

print(f'\nDone in {(time.time()-t0)/60:.1f} min, {len(results):,} results.')

## 8 — Save results & sample profiles

In [ ]:
with open(RESULTS_PKL, 'wb') as fh:
    pickle.dump({'radii': radii, 'bins_lims': BINS_LIMS,
                 'nboot': NBOOT, 'radius_mpc': RADIUS_MPC,
                 'selection': 'wazp_real_data + DELTA_M_I > 0',
                 'results': results}, fh)
print(f'Wrote {len(results):,} results to {RESULTS_PKL}')

errs   = [r for r in results if 'error'  in r]
empty  = [r for r in results if r.get('sigmas') is None]
good   = [r for r in results if r.get('sigmas') is not None]
print(f'  good profiles : {len(good):,}')
print(f'  empty (no src): {len(empty):,}')
print(f'  errors        : {len(errs):,}')
for r in errs[:3]:
    print('   ', r)

# Outer-bin boost should fall near 1 if the field background dominates.
if good:
    b_outer = np.array([r['boosts'][-1] for r in good])
    print(f'\nOuter-bin boost over {len(good):,} clusters:')
    print(f'  median = {np.median(b_outer):.3f}  mean = {b_outer.mean():.3f}  '
          f'std = {b_outer.std():.3f}')


In [ ]:
# Side-by-side: per-cluster ΔΣ profiles and per-cluster boost factors.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── ΔΣ ───────────────────────────────────────────────────────────────────
ax = axes[0]
for r in good[:1000]:
    ax.plot(radii, np.abs(r['sigmas']), color='k', alpha=0.04, lw=0.8)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'$R$  [Mpc, proper]')
ax.set_ylabel(r'$|\Delta\Sigma|$  [$M_\odot/\mathrm{pc}^2$]')
ax.set_title(f'Per-cluster $|\Delta\Sigma|$ profiles (first {min(1000, len(good)):,})')
ax.grid(alpha=0.3, which='both')

# ── boost ────────────────────────────────────────────────────────────────
ax = axes[1]
for r in good[:1000]:
    ax.plot(radii, r['boosts'], color='k', alpha=0.04, lw=0.8)
ax.axhline(1.0, color='crimson', lw=1.0, ls='--')
ax.set_xscale('log')
ax.set_xlabel(r'$R$  [Mpc, proper]')
ax.set_ylabel(r'Boost factor  $B(R) = N_{\rm obs}/N_{\rm random}$')
ax.set_title(f'Per-cluster boosts (first {min(1000, len(good)):,}); should → 1 at large $R$')
ax.grid(alpha=0.3, which='both')

plt.tight_layout()
plt.show()
